# RAFT · Stage 3 — Deploy the fine-tuned model

Deploys the fine-tuned model on the **Developer tier** (no hourly hosting fee). Terraform
(`infra/terraform`, var `raft_student_ft_model_id`) is the reproducible path; this notebook is
the interactive equivalent and the basis for `redeploy_student.ps1`.

**Runtime:** < 10 min. **Cost:** Developer tier has no hourly fee but is **auto-removed after
24 h** — it must be recreated on the morning of the demo (see the day-of checklist). A customised
model permits only **one** deployment at a time.

Deploying requires the **Foundry Owner** role (or `deployments/write`).

In [ ]:
ft_model_id = ""  # e.g. gpt-4.1-mini.ft-<jobid>; if empty, read data/ft_model_id.txt from Stage 2
deployment_name = "raft-student"
deployment_tier = "Developer"  # no hourly hosting fee; auto-removed after 24h
capacity = 20

In [ ]:
import os, time, pathlib
t0 = time.time()
if not ft_model_id:
    ft_model_id = pathlib.Path("data/ft_model_id.txt").read_text(encoding="utf-8").strip()
assert ft_model_id, "ft_model_id required (parameter or data/ft_model_id.txt from Stage 2)"
print(f"Stage 3 · deploying {ft_model_id} as '{deployment_name}' (tier {deployment_tier})")
print("WARNING: Developer-tier deployments are auto-removed after 24h. Redeploy on demo morning.")

In [ ]:
# Deploy via the Cognitive Services management plane (same shape as the Terraform resource).
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

SUB = os.environ["AZURE_SUBSCRIPTION_ID"]
RG = os.environ["AZURE_RESOURCE_GROUP"]
ACCOUNT = os.environ["AI_FOUNDRY_NAME"]
mgmt = CognitiveServicesManagementClient(DefaultAzureCredential(), SUB)
poller = mgmt.deployments.begin_create_or_update(
    RG, ACCOUNT, deployment_name,
    {"sku": {"name": deployment_tier, "capacity": capacity},
     "properties": {"model": {"format": "OpenAI", "name": ft_model_id}}},
)
poller.result()
print(f"Stage 3 done in {time.time()-t0:.0f}s. Deployment '{deployment_name}' ready.")
print(f"Set VITE_RAFT_STUDENT_DEPLOYMENT={deployment_name} and VITE_RAFT_ENABLED=true, then 4_eval.ipynb")